In [636]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Input
import statsmodels.api as sm
from linearmodels.iv import IV2SLS
import seaborn as sns
from itertools import product

## Define prossessing functions

In [638]:
"""
Load data and configs function
"""
# Load  data from file
def load_data(file_path) -> pd.DataFrame:
    """
    Load car data from a CSV, JSON, or Excel file.
    
    Args:
        file_path (str): Path to the file.
        
    Returns:
        pd.DataFrame: Loaded data as a pandas DataFrame.
    """
    if file_path.endswith('.csv'):
        data = pd.read_csv(file_path)
    elif file_path.endswith('.json'):
        data = pd.read_json(file_path)
    elif file_path.endswith('.xlsx'):
        data = pd.read_excel(file_path)
    else:
        raise ValueError("Unsupported file format. Please provide a .csv, .json, or .xlsx file.")
    return data

In [ ]:

def process_dates(data) -> pd.DataFrame:
    """
    Process selling dates and extract selling year, selling month and selling week-of-the-year
    """
    df = data.copy()

    # Clean the string (remove the part in parentheses)
    df['saledate'] = df['saledate'].str.replace(r'\s*\(.*?\)', '', regex=True).str.strip()

    # Convert to datetime (handles GMT offsets) - invalid dates will become NaT
    df['saledate'] = pd.to_datetime(df['saledate'], utc=True, errors='coerce')

    # Exclude rows with invalid dates
    nb_invalids = df[df['saledate'].isna()].shape[0]

    if nb_invalids>0:
        print(nb_invalids, 'rows with invalid dates have been dropped')

    df = df[~df['saledate'].isna()]

    #Extract selling year, months and weeks
    df['sellingyear'] = df['saledate'].map(lambda x: x.year)
    df['sellingmonth'] = df['saledate'].map(lambda x: x.month)
    #df['sellingweek'] = df['saledate'].map(lambda x: x.dt.isocalendar().week)
    df['sellingweek'] = df['saledate'].map(lambda x: x.isocalendar().week)
    return df


In [ ]:
def define_market(data, market_vars, market_label='market', market_code='marketid', minsize=20) -> pd.DataFrame:
    """
    Build two new column (marketid, market) for the market IDs
    
    Args:
        - market_vars (list of str): the names of columns in data that define the market
        - minsize (int): minimum market size to allow. Sales in markets with lower sizes will be dropped
    Returns:
        - The dataframe data with two new columns:
            market: a concatenate of values from variables in market_vars
            marketid: numerical code of market

    Notes:
        - The same function can be used to define products
        - process_dates should be run before this function (to keep all products, set minsize=1)
    """    
    df = data.copy()
    df['market'] = df[market_vars[0]].astype('str')
    
    for var in market_vars[1:]:
        df['market'] = df['market'] + '_' + df[var].astype('str')

    markets_tab = df['market'].value_counts()

    selected_markets = markets_tab.iloc[np.where(markets_tab>=minsize)].index

    nb_droppedmarkets = np.sum(~(markets_tab>=minsize))

    nb_droppedrows = np.sum(~df['market'].isin(selected_markets))

    df = df[df['market'].isin(selected_markets)]

    print(f"\n{nb_droppedmarkets} markets with sizes lower than {minsize} have been dropped; 
    making {nb_droppedrows} dropped rows \n")

    idmarket_mask = dict([(i, selected_markets[i]) for i in range(len(selected_markets))])
    marketid_mask = dict([(selected_markets[i], i) for i in range(len(selected_markets))])
    
    df['marketid'] = df['market'].map(marketid_mask)
    return df

In [ ]:
def recode_categories(series, replacement_mask=dict(),  min_frequency=10000) -> pd.Series:
    """
    recode values of a series of strings
    
    Args:
        - series (pd.Series): series of strings in lower cases.
        - replacement_mask (dict): a dictionary that maps some values to be replaced to their new values
        - min_frequency (int): minimum frequency per category to allow
                        all values with a lower frequency will be recoded as 'other'
    Returns:
        - pd.Series
    """
    s = series.copy()        
    s = s.replace(replacement_mask)
    s = s.map(lambda x: str(x).lower())
    frequencies = s.value_counts()
    popular_values = set(frequencies[frequencies>= min_frequency].index)-{''}
    s = s.map(lambda x: x if x in popular_values else 'other')
    return s

In [ ]:
def order_unorder_categorical_var(data, dropna=True, ordered=[], unordered=[], 
                               ordered_masks=dict(), unordered_masks=dict(), 
                               ordered_min_frequencies=dict(), unordered_min_frequencies=dict()) -> pd.DataFrame:
    """
    Extract and process relevant variables
    Args:
        - data (pd.DataFrame): shouldcontaain a 'state' column
        - numerical, ordered, unordered (list of str): the names of relevant columns that will be set as numerical, ordered categorical and unordered categorical 
        - ordered_masks, unordered_masks (dict of dict): dictionaries of replacement masks to be used as arguments in the recode_categories function for each relevant categorical variable
        - ordered_min_frequencies, unordered_min_frequencies (dict of int): dictionaries that provides the min_frequency argument for the recode_categories function for each relevant categorical variable

    Returns:
        - pd.DataFrame: contains the provided relevant columns along with marketvar and productvar columns
    
    """
    df = data.copy()    

    for var in ordered:
        df[var] = recode_categories(
            df[var], replacement_mask=ordered_masks[var], min_frequency=ordered_min_frequencies[var]
        )
        df[var] = pd.Categorical(df[var], ordered=True)

    for var in unordered:
          df[var] = recode_categories(
            df[var], replacement_mask=ordered_masks[var], min_frequency=ordered_min_frequencies[var]
        )
        df[var] = pd.Categorical(df[var], ordered=False)
    return df


def one_hot_encoder(data, categorical, exog, endog):
    """
    One-hot encode categorical variables and updates the list of
    endogenous and exogenous variables with the dummies and removes the main category keeping track 
    of it with excluded_main_categories.
    """
    df = data.copy()
    excluded_main_categories = []
    for var in categorical:
        # Identify the most populated category for the variable
        main_category = data[var].value_counts().idxmax()
        excluded_main_categories.append(main_category)
        # One-hot encode the variable
        encoded_df = pd.get_dummies(data[var], prefix=var, drop_first=True)

        # Remove the column corresponding to the main category
        main_category_column = f"{var}_{main_category}"

        encoded_df = encoded_df.drop(columns=[main_category_column])

        # Add the remaining encoded columns to the main dataframe
        df = pd.concat([df, encoded_df], axis=1)
        
        # Update exog and endog variables
        dummy_var = list(set(df.columns) - set(data.columns))
        if var in exog:
            exog.remove(var)
            exog = exog + dummy_var
        if var in endog:   
            endog.remove(var)
            endog = endog + dummy_var
    return df, endog, exog, excluded_main_categories


def preprocess_color_interior(df):
    """
    Preprocess the 'color' and 'interior' columns in the DataFrame.
    """
    for var in ['color', 'interior']:
        if var in df.columns:
            df[var].replace({'—':np.nan})  
    return df     


def change_to_numerical(data, numerical):
    """"
    Change the type of the variables in numerical to float
    """
    df = data.copy()
    for var in numerical:
        df[var] = df[var].astype('float')    
    return df

def subset_var_of_interest(data, var_of_interest, marketvar='marketid', productvar='make'):
    """
    Subset the data to keep only the variables of interest
    """
    df = data.copy()
    print('\nNumber of missing per relevant variable in the remaining dataframe \n', np.sum(df.isna(), axis=0))
    print(f"\nA total of {np.sum(np.any(df.isna(), axis=1))} rows with missing data in relevant variables have been dropped \n")
    if dropna:
        df = df.dropna()
    df = df[[marketvar, productvar] + var_of_interest]
    return df

In [643]:
def reduce_to_full_rank(A, tol=1e-10):
    """
    extract a full-rank matrix from A that has same rank as A, by dropping collinear columns
    
    Args: 
        - A (2D array)
    
    Returns: 
        - 2D array: a submatrix extracted from A
    """
    Q, R = np.linalg.qr(A) #QR decomposition
    independent = np.abs(np.diag(R)) > tol
    return A[:, independent]

def get_non_collinear_instruments(Z, X_exog = None, tol=1e-10):
    """
    extract a full-rank matrix from Z whose columns are not collinear with other columns in Z and X_exog
    
    Args:
        - Z (2D array): e.g. matrix of instruments
        - X_exog (2D array): e.g. matrix of included exogenous variables
        - tol (float): the tolerance used to check if the regression SSR is equal to zero
    Returns:
        - 2D array: a submatrix extracted from Z

    Notes:
        - The idea is to regress iteratively each column from Z on other columns in Z and columns in X_exog, and suppress the dependant column from Z if there is a perfect fit
    """

    #Initialize the matrix X of regressor columns with X_exog (and include the constant column)
    if np.all(X_exog==None):
        X = np.ones((Z.shape[0], 1))
    else:
        X = np.asarray(X_exog)
        X = np.hstack([X, np.ones((X.shape[0], 1))])

    keep = []
    NewZ = Z
    for col in Z.columns:
        #get the dependant column
        y = Z[col].values 
        #update the matrix of regressor columns with remaining columns in Z
        Xcol0 = np.hstack([NewZ.drop([col],axis=1), X])
        #get full-rank version of the matrix of regressor columns
        Xcol = reduce_to_full_rank(Xcol0)
        #fit depenant column on regressor columns
        beta = np.linalg.lstsq(Xcol, y, rcond=None)[0]
        y_hat = Xcol @ beta
        residual = y - y_hat    
        if np.linalg.norm(residual) > tol:
            keep.append(col)
        else:
            NewZ = NewZ.drop([col],axis=1)

    return Z[keep], keep

In [ ]:
def aggregate_data(data, pop_data, marketvar='marketid', productvar= 'make', twodegree_polynomial_instruments=False, 
                   aggfunc='mean', numerical=[], categorical=[], dep=[], endog=[], exog=[]):
    """
    aggregate the data at (market, product)-level

    Args:
        - data (pd.DataFrame): dataset of cars
        - pop_data: dataset of population sizes; contains at least following columns: 'state' and 'population (2015)'
        - marketvar, productvar (str): column names for market, product
        - twodegree_polynomial_instruments (bool): set value 'True' to obtain BLP instruments from 2-degree polynomial basis of exogenous characteristics
        - aggfunc (str): the aggregation function (preferably 'mean' or 'median')
        - numerical, categorical (list of str): the names of data columns that are numerical, categorical
        - dep, endog, exog (list of str): the names of data columns that are set as dependant variables, endogenous explanatory variables, enxogenous explanatory variables
    
    Returns : 
        - pd.DataFrame
        - list: names of dependant variables or variables in the returned dataframe that can be used to compute dependant variables 
        - list: names of endogenous explanatory variables in the returned dataframe
        - list: names of exogenous explanatory variables in the returned dataframe
        - list: names of excluded instruments in the returned dataframe

    Notes: 
        - the returned excluded instruments are the BLP instruments for price (i.e. sum of rival products' characteristics). see Berry, Levingson and Pakes, 1995.
        - 'state' column is needed inside data as key to get market sizes (proxied by state-varying population sizes of year 2015) from the pop_data dataframe
    """
    
    #Extract categories for each categorical variable (the most frequent category is excluded)
    Categories = [(var, cat) for var in categorical for cat in data[var].value_counts().index[1:]]

    df = data[[marketvar, productvar, 'state'] + numerical].copy()
    pop_df = pop_data.copy()
    depvars = []; endogvars = []; exogvars = []
    
    for var in numerical:
        if var in dep:
            depvars = depvars + [var]
        if var in endog:
            endogvars = endogvars + [var]
        if var in exog:
            exogvars = exogvars + [var]

    for var, cat in Categories:
        varcat = f"{var}_{cat}"
        df[varcat] = (data[var] == cat).astype(float)
        if var in dep:
            depvars = depvars + [varcat]
        if var in endog:
            endogvars = endogvars + [varcat]
        if var in exog:
            exogvars = exogvars + [varcat]
            
    df = df.groupby([marketvar, productvar,'state'], observed=True).agg(aggfunc).reset_index()
    df['sales'] = data.groupby([marketvar, productvar,'state'], observed=True).size().values

    pop_df['population (2015)'] = pop_df['population (2015)'].map(lambda x: x.replace(',','')).astype('float')
    outsideoption_df = df[[marketvar,'state','sales']].groupby(by=[marketvar,'state'],observed=True).sum().reset_index().rename({'sales':'allsales'},axis=1)
    pop_df = pop_df.merge(outsideoption_df,on='state')
    df = pop_df.merge(df, on=[marketvar,'state'])
    df['share'] = df['sales']/df['population (2015)']
    df['share_oo'] = 1-(df['allsales']/df['population (2015)'])
    df['log_share_ratio'] = np.log(df['share']/df['share_oo'])

    depvars = depvars + ['population (2015)','allsales', 'share_oo', 'sales', 'share', 'log_share_ratio']

    #Extract instruments
    if twodegree_polynomial_instruments==False:
        Z = df[[marketvar]+exogvars].groupby([marketvar]).apply(
            lambda x: x.assign(**dict(
                [('Nb_RivalProducts',x.shape[0]-1)]+[(var+'_RivalProducts', x[var].sum()-x[var]) for var in exogvars]
            )), include_groups=False
        ).reset_index(drop=True).drop(exogvars, axis=1)
    if twodegree_polynomial_instruments==True: 
        Z = df[[marketvar]+exogvars].groupby([marketvar]).apply(
            lambda x: x.assign(**dict(
                [('Nb_RivalProducts',x.shape[0]-1)]+[(var+'_RivalProducts', x[var].sum()-x[var]) for var in exogvars]+\
                [(var1+'*'+var2+'_RivalProducts', (x[var1]*x[var2]).sum()-x[var1]*x[var2]) for var1, var2 in list(product(exogvars, repeat=2))]
            )), include_groups=False
        ).reset_index(drop=True).drop(exogvars, axis=1)
    #reduce instruments to collinearity-proof instruments
    Z, instrvars = get_non_collinear_instruments(Z, df[exogvars])
    df = df[[marketvar,productvar,'state']+depvars+endogvars+exogvars].reset_index(drop=True).join(Z.reset_index(drop=True))  
    
    return df , depvars, endogvars, exogvars, instrvars
    

In [ ]:


def compute_market_shares(df, pop_df, marketvar, productvar):
    """
    Compute market shares and related variables.
    """
    outsideoption_df = df[[marketvar, 'state', 'sales']].groupby(
        by=[marketvar, 'state'], observed=True
    ).sum().reset_index().rename({'sales': 'allsales'}, axis=1)
    pop_df = pop_df.merge(outsideoption_df, on='state')
    df = pop_df.merge(df, on=[marketvar, 'state'])
    df['share'] = df['sales'] / df['population (2015)']
    df['share_oo'] = 1 - (df['allsales'] / df['population (2015)'])
    df['log_share_ratio'] = np.log(df['share'] / df['share_oo'])
    return df


def extract_instruments(df, exogvars, marketvar, twodegree_polynomial_instruments):
    """
    Extract instruments for the model.
    """
    if not twodegree_polynomial_instruments:
        Z = df[[marketvar] + exogvars].groupby([marketvar]).apply(
            lambda x: x.assign(**dict(
                [('Nb_RivalProducts', x.shape[0] - 1)] +
                [(var + '_RivalProducts', x[var].sum() - x[var]) for var in exogvars]
            )), include_groups=False
        ).reset_index(drop=True).drop(exogvars, axis=1)
    else:
        Z = df[[marketvar] + exogvars].groupby([marketvar]).apply(
            lambda x: x.assign(**dict(
                [('Nb_RivalProducts', x.shape[0] - 1)] +
                [(var + '_RivalProducts', x[var].sum() - x[var]) for var in exogvars] +
                [(var1 + '*' + var2 + '_RivalProducts', (x[var1] * x[var2]).sum() - x[var1] * x[var2])
                 for var1, var2 in list(product(exogvars, repeat=2))]
            )), include_groups=False
        ).reset_index(drop=True).drop(exogvars, axis=1)
    return Z


def aggregate_data(data, pop_data, marketvar='marketid', productvar='make', twodegree_polynomial_instruments=False,
                   aggfunc='mean', numerical=[], categorical=[], dep=[], endog=[], exog=[]):
    """
    Aggregate the data at (market, product)-level.
    """
    df = data[[marketvar, productvar, 'state'] + numerical].copy()
    pop_df = pop_data.copy()
    # Extract categories and initialize variables
    excluded_main_categories = []
    
    for var in categorical:
        results = one_hot_encoder(df, var, exog, endog)
        df = results[0]; endogvars = endogvars + results[1]; exogvars = exogvars + results[2]
        excluded_main_categories = excluded_main_categories.append(results[3])  

    # I am here
    # Aggregate data and compute sales
    df = df.groupby([marketvar, productvar, 'state'], observed=True).agg(aggfunc).reset_index()
    df['sales'] = data.groupby([marketvar, productvar, 'state'], observed=True).size().values

    # Process population data and compute market shares
    pop_df['population (2015)'] = pop_df['population (2015)'].map(lambda x: x.replace(',', '')).astype('float')
    df = compute_market_shares(df, pop_df, marketvar, productvar)

    # Update dependent variables
    depvars += ['population (2015)', 'allsales', 'share_oo', 'sales', 'share', 'log_share_ratio']

    # Extract instruments
    Z = extract_instruments(df, exogvars, marketvar, twodegree_polynomial_instruments)

    # Reduce instruments to collinearity-proof instruments
    Z, instrvars = get_non_collinear_instruments(Z, df[exogvars])

    # Finalize the dataframe
    df = df[[marketvar, productvar, 'state'] + depvars + endogvars + exogvars].reset_index(drop=True).join(Z.reset_index(drop=True))

    return df, depvars, endogvars, exogvars, instrvars

In [ ]:
def extract_relevant_variables(data, marketvar='marketid', productvar='make', dropna=True,
                               numerical=[], ordered=[], unordered=[], 
                               ordered_masks=dict(), unordered_masks=dict(), 
                               ordered_min_frequencies=dict(), unordered_min_frequencies=dict()) -> pd.DataFrame:
    """
    Extract and process relevant variables
    
    Args:
        - data (pd.DataFrame): shouldcontaain a 'state' column
        - marketvar (str): the column name in data for market
        - productvar (str): the column name in data for product 
        - numerical, ordered, unordered (list of str): the names of relevant columns that will be set as numerical, ordered categorical and unordered categorical 
        - ordered_masks, unordered_masks (dict of dict): dictionaries of replacement masks to be used as arguments in the recode_categories function for each relevant categorical variable
        - ordered_min_frequencies, unordered_min_frequencies (dict of int): dictionaries that provides the min_frequency argument for the recode_categories function for each relevant categorical variable

    Returns:
        - pd.DataFrame: contains the provided relevant columns along with marketvar and productvar columns

    Notes:
        - The distinction in the type of variables (i.e., numerical, ordered, unordered) can be usefull in a later update that considers missing value imputation using chained equations
        - There is a particular processing for 'color', 'interior' regarding value '—' 
            (which isn't recongnized as a 'minus' symbol in some computers)
    
    """
    df = data.copy()
    if productvar in numerical+ordered+unordered:
        df = df[[marketvar]+['state']+numerical+ordered+unordered]
    else:
        df = df[[marketvar,productvar]+numerical+ordered+unordered]
    for var in ['color', 'interior']:
        if var in df.columns:
            df[var].replace({'—':np.nan})           
    print('\nNumber of missing per relevant variable in the remaining dataframe \n', np.sum(df.isna(), axis=0))
    print(f"\nA total of {np.sum(np.any(df.isna(), axis=1))} rows with missing data in relevant variables have been dropped \n")
    if dropna:
        df = df.dropna()
    for var in ordered:
        df[var] = df[var].map(lambda x: str(x).lower())
        df[var] = recode_categories(
            df[var], replacement_mask=ordered_masks[var], min_frequency=ordered_min_frequencies[var]
        )
        df[var] = pd.Categorical(df[var], ordered=True)
    for var in unordered:
        df[var] = df[var].map(lambda x: str(x).lower())
        df[var] = recode_categories(
            df[var], replacement_mask=unordered_masks[var], min_frequency=unordered_min_frequencies[var]
        )
        df[var] = pd.Categorical(df[var], ordered=False)
    for var in numerical:
        df[var] = df[var].astype('float')
    return df

## Deploy prossessing functions

In [ ]:

#import the car sales and population size datasets
cars_file_path = '/Users/colin/Library/CloudStorage/OneDrive-UniversitedeMontreal/On_Going_Studies/IVADO_Project/car_prices.csv'
population_file_path = '/Users/colin/Library/CloudStorage/OneDrive-UniversitedeMontreal/On_Going_Studies/IVADO_Project/states_populations_yr2015.csv'
df = load_data(cars_file_path)
pop_df = load_data(population_file_path)
#process dates and define markets
df = process_dates(df)
df = define_market(df, market_vars=['sellingyear','sellingmonth','state'], market_label='market', market_code='marketid', minsize=20)
#value '—' for df columns color and interior is not recognized as a minus character [('—' == '-') gives False value]. So we change it manually to missing value 
df['color'] = df['color'].replace({'—':np.nan})
df['interior'] = df['interior'].replace({'—':np.nan})
#prepare replacing masks for the extract_relevant_variables function
make_mask = {
    'gmc truck':'gmc', 'dodge tk':'dodge', 'mazda tk':'mazda', 'hyundai tk':'hyundai', 'mercedes-b':'mercedes-benz',
    'chev truck':'chevrolet', 'ford tk':'ford', 'ford truck':'ford', 'vw':'volkswagen'
}
body_mask = body_type_map = {
    #suvs
    'suv': 'suv',      
    #sedans
    'sedan': 'sedan', 'g sedan': 'sedan', 'elantra coupe': 'sedan',
    #coupes
    'coupe': 'coupe', 'g coupe': 'coupe', 'genesis coupe': 'coupe', 'cts coupe': 'coupe', 'koup': 'coupe', 'cts-v coupe': 'coupe',
    'g37 coupe': 'coupe', 'q60 coupe': 'coupe',
    # convertibles
    'convertible': 'convertible', 'g convertible': 'convertible', 'g37 convertible': 'convertible', 'q60 convertible': 'convertible',
    'beetle convertible': 'convertible', 'granturismo convertible': 'convertible',
    # wagons
    'wagon': 'wagon', 'cts wagon': 'wagon', 'tsx sport wagon': 'wagon', 'cts-v wagon': 'wagon',
    # hatchbacks
    'hatchback': 'hatchback',
    # vans / minivans
    'minivan': 'van', 'van': 'van', 'e-series van': 'van', 'promaster cargo van': 'van', 'ram van': 'van', 'transit van': 'van',
    # pickup trucks - cab variants
    'crew cab': 'pickup', 'double cab': 'pickup', 'crewmax cab': 'pickup', 'access cab': 'pickup', 'king cab': 'pickup',
    'supercrew': 'pickup', 'extended cab': 'pickup', 'supercab': 'pickup', 'regular cab': 'pickup', 'regular-cab': 'pickup',
    'quad cab': 'pickup', 'club cab': 'pickup', 'xtracab': 'pickup', 'mega cab': 'pickup', 'cab plus': 'pickup', 'cab plus 4': 'pickup'
}

color_mask = dict()
interior_mask = dict()
#prepare minimum frequencies for the extract_relevant_variables function
make_freq = 1; body_freq = 10000; year_freq = 1; color_freq = 10000; interior_freq = 10000;
#extract relevant variables
unordered_masks = dict([('year',year_mask),('make',make_mask),('body',body_mask),('color',color_mask),('interior',interior_mask)])
unordered_minfreq = dict([('year',1),('make',10000),('body',1),('color',10000),('interior',10000)])
df = extract_relevant_variables(
    df, marketvar='marketid', productvar='make', dropna=True, 
    numerical=['sellingprice','odometer','condition'], ordered=[], unordered=['year','make','body','color','interior'],
    ordered_masks=dict(), unordered_masks=unordered_masks,  ordered_min_frequencies=dict(), unordered_min_frequencies=unordered_minfreq
)
#extract the final dataframe and the names of dependant, endogenous, exogenous and instruments
df,  depvars, endogvars, exogvars, instrvars = aggregate_data(
    df, pop_df, marketvar='marketid', productvar='make', twodegree_polynomial_instruments=False, 
    aggfunc='mean', numerical=['sellingprice','odometer','condition'], 
    categorical=['year','make','body','color','interior'], dep=[], 
    endog=['sellingprice'], exog=['odometer','condition','year','body','color','interior']
)

38 rows with invalid dates have been dropped

63 markets with sizes lower than 20 have been dropped; making 485 dropped rows 


Number of missing per relevant variable in the remaining dataframe 
 marketid            0
state               0
sellingprice        0
odometer           93
condition       11794
year                0
make            10282
body            13174
color           25416
interior        17808
dtype: int64

A total of 60818 rows with missing data in relevant variables have been dropped 



In [120]:
depvars

['population (2015)',
 'allsales',
 'share_oo',
 'sales',
 'share',
 'log_share_ratio']

In [121]:
endogvars

['sellingprice']

In [122]:
np.array(exogvars)

array(['odometer', 'condition', 'year_2013', 'year_2014', 'year_2011',
       'year_2008', 'year_2007', 'year_2010', 'year_2006', 'year_2009',
       'year_2005', 'year_2004', 'year_2003', 'year_2015', 'year_2002',
       'year_1999', 'year_2001', 'year_2000', 'body_suv', 'body_pickup',
       'body_van', 'body_hatchback', 'body_coupe', 'body_wagon',
       'body_convertible', 'color_white', 'color_gray', 'color_silver',
       'color_blue', 'color_red', 'color_other', 'color_gold',
       'color_green', 'interior_gray', 'interior_beige', 'interior_tan',
       'interior_other'], dtype='<U16')

In [123]:
np.array(instrvars)

array(['Nb_RivalProducts', 'odometer_RivalProducts',
       'condition_RivalProducts', 'year_2013_RivalProducts',
       'year_2014_RivalProducts', 'year_2011_RivalProducts',
       'year_2008_RivalProducts', 'year_2007_RivalProducts',
       'year_2010_RivalProducts', 'year_2006_RivalProducts',
       'year_2009_RivalProducts', 'year_2005_RivalProducts',
       'year_2004_RivalProducts', 'year_2003_RivalProducts',
       'year_2015_RivalProducts', 'year_2002_RivalProducts',
       'year_1999_RivalProducts', 'year_2001_RivalProducts',
       'year_2000_RivalProducts', 'body_suv_RivalProducts',
       'body_pickup_RivalProducts', 'body_van_RivalProducts',
       'body_hatchback_RivalProducts', 'body_coupe_RivalProducts',
       'body_wagon_RivalProducts', 'body_convertible_RivalProducts',
       'color_white_RivalProducts', 'color_gray_RivalProducts',
       'color_silver_RivalProducts', 'color_blue_RivalProducts',
       'color_red_RivalProducts', 'color_other_RivalProducts',
       'c

In [124]:
df

,marketid,make,state,population (2015),allsales,share_oo,sales,share,log_share_ratio,sellingprice,...,color_silver_RivalProducts,color_blue_RivalProducts,color_red_RivalProducts,color_other_RivalProducts,color_gold_RivalProducts,color_green_RivalProducts,interior_gray_RivalProducts,interior_beige_RivalProducts,interior_tan_RivalProducts,interior_other_RivalProducts
0,1,bmw,ca,39144818.0,19533,0.999501,1381,0.000035,-10.251716,20901.973208,...,2.359580,1.379342,1.230934,0.848863,0.403178,0.299384,4.498197,2.316830,1.090180,0.353838
1,1,chevrolet,ca,39144818.0,19533,0.999501,1766,0.000045,-10.005807,11546.932616,...,2.301496,1.406730,1.143933,0.820490,0.381559,0.299497,4.233706,2.419035,1.199667,0.449101
2,1,chrysler,ca,39144818.0,19533,0.999501,409,0.000010,-11.468564,8861.980440,...,2.353351,1.377189,1.148637,0.806325,0.347401,0.301830,4.360799,2.390792,1.148410,0.462615
3,1,dodge,ca,39144818.0,19533,0.999501,812,0.000021,-10.782779,10424.014778,...,2.322279,1.359655,1.130512,0.828838,0.367170,0.305375,4.392501,2.428285,1.199268,0.461742
4,1,ford,ca,39144818.0,19533,0.999501,2705,0.000069,-9.579422,12776.851017,...,2.356376,1.405048,1.163570,0.830793,0.395911,0.279563,4.231676,2.359021,1.135526,0.440468
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3513,217,dodge,ns,943000.0,31,0.999967,4,0.000004,-12.370494,4825.000000,...,1.666667,1.333333,0.000000,0.333333,0.000000,0.000000,3.583333,0.000000,1.583333,0.000000
3514,217,ford,ns,943000.0,31,0.999967,4,0.000004,-12.370494,6400.000000,...,1.666667,1.333333,0.000000,0.333333,0.000000,0.000000,2.583333,0.000000,2.583333,0.000000
3515,217,hyundai,ns,943000.0,31,0.999967,12,0.000013,-11.271882,8216.666667,...,1.666667,1.333333,0.000000,0.333333,0.000000,0.000000,3.583333,0.000000,1.583333,0.000000
3516,217,nissan,ns,943000.0,31,0.999967,2,0.000002,-13.063642,8900.000000,...,1.000000,1.000000,0.000000,0.333333,0.000000,0.000000,3.250000,0.000000,2.583333,0.000000
